# ECON2041 Week 8 tutorial: bedrooms, areas, and one interaction

### What you'll be able to do by the end

- Add a second continuous explanatory variable and compare the guests coefficient with and without it
- Use `C()` to turn the area column into a block of dummies and read each coefficient as a gap from the reference category
- Change the reference category and say which estimates differ and which are identical
- Fit an interaction and write out the two fitted lines it implies

### Different types of cells

- 🟢 Read and run: these cells have pre-written code, you can simply run them, read and interpret the output
- ✏️ You write: your turn to try to write code or sometimes add a short written answer
- 🤔 You think: stop and think on your own before running the next cell or reading on
- 💬 Discuss: talk with your neighbors before typing or running
- ⭐ Optional: extra exercise if you have time

## Setup

Our usual setup block, the same one as last week.

In [ ]:
# 🟢 Read and run: our standard ECON2041 setup block
import numpy as np               # numerical tools (nicknamed np)
import pandas as pd              # data tools (nicknamed pd)
import matplotlib.pyplot as plt  # plotting tools (nicknamed plt)
import seaborn as sns            # statistical charts (nicknamed sns)
from statsmodels.formula.api import ols  # ordinary least squares regression

# Keep scalar output plain under NumPy 2 (0.5, not np.float64(0.5)).
if np.lib.NumpyVersion(np.__version__) >= "2.0.0":
    np.set_printoptions(legacy="1.25")

DATA = "https://emiliatjernstrom.com/econ2041/data"   # Unit datasets live at this web address

print("Setup done!")

## Back to the Sydney Airbnb listings

We'll continue with last week's file and question:

> What makes a Sydney Airbnb listing expensive?

Source: [Inside Airbnb](https://insideairbnb.com/), Sydney, snapshot of 16 June 2026, CC BY 4.0.

This should feel familiar, as we start from the parallel-lines model from last week's tutorial: $\widehat{\text{price}} = 16.1 + 63.8 \times \text{accommodates}$ for rooms and $\widehat{\text{price}} = 135.0 + 63.8 \times \text{accommodates}$ for entire homes.

The three cells below load the file, build the `entire` dummy, and fit that model. Let's run those first!

In [ ]:
# 🟢 Read and run: load sydney-airbnb-teaching.csv into a dataframe called airbnb and look at the first few rows
airbnb = pd.read_csv(f"{DATA}/sydney-airbnb-teaching.csv")

airbnb.head()

In [ ]:
# 🟢 Read and run: build the entire dummy from last week's tutorial
#                  this takes a value of 1 for an entire home or apartment
#                  and a value of 0 for any kind of "room"
airbnb["entire"] = (airbnb["room_type"] == "Entire home/apt").astype(int)

airbnb["entire"].mean().round(2)

In [ ]:
# 🟢 Read and run: fit the parallel-lines model that we estimated last week
#                   `price` on `accommodates` and `entire`
multi = ols("price ~ accommodates + entire", data=airbnb).fit()

multi.params.round(1)

## Add bedrooms as a second continuous variable

### Question 1: how much of the per-guest difference is really bedrooms?

`bedrooms` is a continuous variable, like `accommodates`, so it enters the formula the same way.
Listings with more bedrooms presumably sleep more guests too, and the correlation between the two says how tightly.

In [ ]:
# 🟢 Read and run: the correlation between the number of guests and the number of bedrooms
airbnb[["accommodates", "bedrooms"]].corr().round(2)

🤔 The correlation is 0.88, so listings with more bedrooms tend to sleep more guests too.

Last week's guests coefficient, 63.8, compared listings of the same room type that differ by 1 guest. Do those two listings usually also differ in bedrooms?

🤔 Holding the number of guests fixed, would you expect a listing with 1 more bedroom to be priced higher or lower?

🤔 So the 63.8 mixed extra guests with the extra bedrooms that usually accompany them.
Guess: once `bedrooms` is in the formula, will the coefficient on `accommodates` be larger or smaller than 63.8?
And which of the two, guests or bedrooms, do you expect to have the larger coefficient?

In [ ]:
# ✏️ You write: run a regression of price on accommodates, bedrooms, and entire together in the airbnb file, store the result as multi_bed,
# and show the coefficients rounded to 1 decimal
# Hint: three explanatory variables to the right of ~, each separated by a +

✏️ Complete the two sentences:

- Holding the number of bedrooms and room type fixed, a listing that sleeps 1 more guest is priced about \$________ higher a night, on average.
- Holding the number of guests and room type fixed, a listing with 1 more bedroom is priced about \$________ higher a night, on average.

🤔 Compare the new guests coefficient with 63.8. Which of your two guesses came true?

💬 With your neighbors: describe two listings that differ by 1 guest but have the same number of bedrooms and the same room type. Is that the comparison you had in mind when you first read the \$63.8?

## Areas as a block of dummies

### Question 2: how much does location matter, holding size and room type fixed?

`neighbourhood` records the listing's area within Sydney, 38 areas across the file.
Area is a category, not a number, so each area needs its own dummy variable. Two ways to create them:

- By hand, with `==` and `.astype(int)`, as we did for `entire`. With 38 areas that is 37 dummy variables, one line each
- With `C()`, the command we used in the live lecture to create dummy variables for the three unemployment groups in a single step

We'll use `C()`. To keep the output readable, we'll keep three areas: Parramatta in the west, Sydney in the city center, and Waverley around Bondi Beach.
The first cell below keeps their listings with `.isin()`, which asks every row whether its area is one of the names in the list, as `==` asks about a single value.

In [ ]:
# 🟢 Read and run: keep the listings in three areas in a smaller dataframe called three, and count them
three = airbnb[airbnb["neighbourhood"].isin(["Parramatta", "Sydney", "Waverley"])]

three["neighbourhood"].value_counts()

In [ ]:
# 🟢 Read and run: mean nightly price in each of the three areas
three.groupby("neighbourhood")["price"].mean().round(1)

In [ ]:
# ✏️ You write: run a regression of price on accommodates, bedrooms, entire, and the area dummies in the dataframe three,
# store the result as areas, and show the coefficients rounded to 1 decimal
# Hint: wrap the area column as C(neighbourhood) in the formula and Python builds the dummies itself;
#       remember data=three, not data=airbnb

🤔 Look at the rows of the output:

- There are three areas in the data but only two `C(neighbourhood)` rows. Which area has no row, and what does the `Intercept` row describe?
- Holding guests, bedrooms, and room type fixed, how much higher is a Waverley listing priced than a Parramatta listing? Compare with the difference between the two raw means above.

In [ ]:
# ✏️ You write: refit the same model with Sydney as the reference category, store the result as areas_syd,
# and show the coefficients rounded to 1 decimal
# Hint: C(neighbourhood, Treatment(reference="Sydney")) picks the reference; the formula string then needs
#       single quotes on the outside, because "Sydney" already uses double quotes

✏️ Complete the sentence: holding guests, bedrooms, and room type fixed, a Parramatta listing is priced about \$________ lower a night than a Sydney listing, on average.

🤔 Compare the two outputs row by row. Which estimates differ and which are identical? Do the two models give a different fitted price for any listing?

## Let the slope differ by room type

### Question 3: does the guests slope differ between rooms and entire homes?

Every model so far gives rooms and entire homes the same slope on guests.
The week 8 live lecture let the slopes differ with an interaction term, the product of the two variables:

$$\text{price}_i = \beta_0 + \beta_1 \, \text{accommodates}_i + \beta_2 \, \text{entire}_i + \beta_3 \, (\text{accommodates}_i \times \text{entire}_i) + u_i$$

What the product does:

- For a room, `entire` $= 0$, so the product is 0 and the term drops out. The room line has intercept $\beta_0$ and slope $\beta_1$
- For an entire home, `entire` $= 1$, so the product equals `accommodates`, and each guest adds $\beta_3$ on top of $\beta_1$. The entire-home line has intercept $\beta_0 + \beta_2$ and slope $\beta_1 + \beta_3$

So $\beta_3$ is the difference in slopes: how much more 1 extra guest is associated with in an entire home than in a room.
A positive $\beta_3$ means the entire-home line is steeper.
We'll go back to the full `airbnb` file for this question.

🤔 Before you run anything: do you expect $\beta_3$ to be positive or negative?
That is, is 1 more guest associated with a larger price difference in an entire home or in a room?
Think about what 1 more guest means in each kind of listing.

In [ ]:
# ✏️ You write: run a regression of price on accommodates, entire, and their interaction in the full airbnb file,
# store the result as inter, and show the coefficients rounded to 1 decimal
# Hint: accommodates * entire expands to accommodates + entire + accommodates:entire

✏️ Fill in the two fitted lines the model implies:

- Rooms (`entire` $= 0$): intercept $\hat{\beta}_0 =$ ________ and slope $\hat{\beta}_1 =$ ________
- Entire homes (`entire` $= 1$): intercept $\hat{\beta}_0 + \hat{\beta}_2 =$ ________ and slope $\hat{\beta}_1 + \hat{\beta}_3 =$ ________

🤔 Was your sign prediction right? Compare the two slopes with the shared slope of 63.8 from the parallel-lines model.

In [ ]:
# 🟢 Read and run: draw the two fitted lines over the scatterplot
b = inter.params
guests = np.arange(1, airbnb["accommodates"].max() + 1)

sns.scatterplot(data=airbnb, x="accommodates", y="price", s=8, alpha=0.2, color="purple")
plt.plot(guests, b["Intercept"] + b["accommodates"] * guests, color="gray", label="Room")
plt.plot(guests, (b["Intercept"] + b["entire"]) + (b["accommodates"] + b["accommodates:entire"]) * guests,
         color="orange", label="Entire home")
plt.xlabel("Guests the listing sleeps")
plt.ylabel("Nightly price (AUD)")
plt.legend()
plt.show()

💬 With your neighbors: last week's model says an entire home is priced \$119 higher than a room at every number of guests. Use your two lines from this week's model to work out the gap at 2 guests and at 6 guests.
Both models are least-squares fits to the same data.
What does the question "how much more does an entire home cost?" need before it has one answer?

### ⭐ Question 4 (optional): all four room types at once

Last week's `entire` dummy put hotel rooms, private rooms, and shared rooms in one group.
`C()` can keep them apart.

In [ ]:
# ✏️ You write (optional): run a regression of price on accommodates and the four room types in the full airbnb file,
# store the result as rooms4, and show the coefficients rounded to 1 decimal
# Hint: C(room_type) builds the dummies from the room-type column, as C(neighbourhood) did above

🤔 Which room type is the reference category, and why that one?
Holding the number of guests fixed, how much lower is a private room priced than an entire home? Compare with the \$119 from last week.

### ⭐ Question 5 (optional): all 38 areas at once

Question 2 kept three areas so the output stayed short.
`C()` handles the full column the same way, so the only change is the dataframe: 38 areas, 37 dummy variables, and one reference category.

In [ ]:
# ✏️ You write (optional): run a regression of price on accommodates, bedrooms, entire, and the area dummies in the full airbnb file,
# store the result as areas_all, and show the coefficients rounded to 1 decimal
# Hint: the same formula as areas, with data=airbnb

🤔 The output has 37 area rows. Ashfield, the first area alphabetically, is the reference, so each row is a gap relative to an Ashfield listing with the same number of guests, bedrooms, and room type.

- Which area has the largest coefficient, and which the smallest? Where in Sydney are they?
- In words, what does the largest coefficient mean?
- Would you find the same two areas if Sydney were the reference instead?